# 05 · Kiểm thử PRDetect V4

Bật GPU và Internet, gắn bộ mã V4 và Dataset checkpoint EMA × A1 cũ đã giải nén. Đặt `CHECKPOINT_DIR` ở ô cấu hình. Mỗi tập dùng chung embedding A1 cho các seed. Sau khi chấm đủ seed, graph tạm được giải phóng.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
from zipfile import ZipFile
from IPython.display import FileLink, display

WORK = Path('/kaggle/working/prdetect_v4')
REPO = WORK / 'PRDetect-Research'
INPUT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=True)
# The datasets and legacy model code used by the five original runs.
if not (REPO / 'evaluate_v3_quang.py').is_file():
    REPO.mkdir(exist_ok=True)
    subprocess.run(['git', 'init', '-q'], cwd=REPO, check=True)
    subprocess.run(['git', 'fetch', '--depth', '1', 'https://github.com/tphan1420/PRDetect-Research.git',
                    '9c8846d5198164124cfb19902fe65fb80e7387c5'], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO, check=True)

files = ('v4.py', 'v3_a1_fusion.py', 'train_v4.py', 'train_v3_dual_best.py')
archives = list(INPUT.rglob('prdetect_v4_code.zip'))
folders = [p.parent for p in INPUT.rglob('train_v4.py') if all((p.parent / f).is_file() for f in files)]
if len(archives) + len(folders) == 1:
    if archives:
        with ZipFile(archives[0]) as z:
            for name in files:
                (REPO / name).write_bytes(z.read(name))
    else:
        for name in files:
            shutil.copyfile(folders[0] / name, REPO / name)
elif archives or folders or not all((REPO / f).is_file() for f in files):
    raise ValueError('Gắn đúng một bản prdetect_v4_code.zip hoặc thư mục đã giải nén vào Kaggle Input.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-v3-eval.txt'),
                'torch-geometric'], check=True)
import spacy
if not spacy.util.is_package('en_core_web_sm'):
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
import torch
assert torch.cuda.is_available(), 'Bật GPU trong Kaggle Settings'
print(torch.cuda.get_device_name(0))

In [ ]:
# Chọn thư mục Dataset chứa 10 tệp .pt đã giải nén và manifest.json.
CHECKPOINT_DIR = Path('/kaggle/input/datasets/your-account/v3-ema-a1-checkpoints')
SEEDS = [2020, 2022, 2024, 2025, 2026]
BATCH_SIZE = 4
OUT = WORK / 'tables'

In [ ]:
subprocess.run([sys.executable, 'v4.py', 'evaluate', '--suite', 'full',
                '--checkpoint-dir', str(CHECKPOINT_DIR), '--seeds', *map(str, SEEDS),
                '--batch-size', str(BATCH_SIZE), '--device', 'cuda', '--output-dir', str(OUT)],
               cwd=REPO, check=True)
import pandas as pd
display(pd.read_csv(OUT / 'summary.csv'))
archive = shutil.make_archive(str(WORK / 'v4_tables_2_3_4'), 'zip', OUT.parent, OUT.name)
print(archive)
display(FileLink(archive))